In [20]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.tsa.statespace.sarimax import SARIMAX

In [21]:
df = pd.read_csv('../data/AEP_hourly.csv')

df['Datetime'] = pd.to_datetime(df['Datetime'])
df = df.groupby('Datetime').mean().reset_index()
df.set_index('Datetime', inplace=True)
df.head()

,AEP_MW
Datetime,
2004-10-01 01:00:00,12379.0
2004-10-01 02:00:00,11935.0
2004-10-01 03:00:00,11692.0
2004-10-01 04:00:00,11597.0
2004-10-01 05:00:00,11681.0


In [22]:
df = df.asfreq('h')
df['AEP_MW'] = df['AEP_MW'].interpolate(method='linear')
print(df.index.freq)

<Hour>


In [23]:
print("Monotonic:", df.index.is_monotonic_increasing)
print("Unique:", df.index.is_unique)
print("Duplicates:", df.index.duplicated().sum())

Monotonic: True
Unique: True
Duplicates: 0


In [34]:
df = df.loc[df.index.max() - pd.Timedelta(days=60): df.index.max()]
len(df)

721

In [35]:
train, test = np.split(df, [int(0.6*len(df))])

/home/iprabhsidhu/EnergyPrediction/venv/lib/python3.12/site-packages/numpy/_core/fromnumeric.py:57: FutureWarning: 'DataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'DataFrame.transpose' instead.
  return bound(*args, **kwds)


In [36]:
params = {
    'non_seasonal': (1,0,2),
    'seasonal':(1,1,0,24),
}
Model = SARIMAX(train,order=params['non_seasonal'],seasonal_order=params['seasonal'])
Model = Model.fit(disp=False, method='powell')

print(Model.summary())

                                      SARIMAX Results                                      
Dep. Variable:                              AEP_MW   No. Observations:                  432
Model:             SARIMAX(1, 0, 2)x(1, 1, [], 24)   Log Likelihood               -2955.766
Date:                             Thu, 12 Feb 2026   AIC                           5921.531
Time:                                     22:03:12   BIC                           5941.588
Sample:                                 07-04-2018   HQIC                          5929.468
                                      - 07-21-2018                                         
Covariance Type:                               opg                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
ar.L1          0.7732      0.007    114.582      0.000       0.760       0.786
ma.L1          0.8821      

In [37]:
pred = Model.get_prediction(
    start=test.index[0],
    end=test.index[-1],
    dynamic=False
)
pred_mean = pred.predicted_mean
pred_mean = pred_mean.reindex(test.index)

In [38]:
pred_mean.index == test.index

array([ True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,

In [39]:
print(pred_mean.head())
print(test.head())

Datetime
2018-07-22 00:00:00    13061.720584
2018-07-22 01:00:00    12376.785782
2018-07-22 02:00:00    11901.612230
2018-07-22 03:00:00    11487.553934
2018-07-22 04:00:00    11403.424039
Freq: h, Name: predicted_mean, dtype: float64
                      AEP_MW
Datetime                    
2018-07-22 00:00:00  13079.0
2018-07-22 01:00:00  12274.0
2018-07-22 02:00:00  11784.0
2018-07-22 03:00:00  11383.0
2018-07-22 04:00:00  11202.0
